In [1]:
%pip install -q --break-system-packages -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import hashlib
import json
import os
import time

os.environ["HF_HOME"] = "models/hf"

import bm25s
import numpy as np
import pandas as pd
import Stemmer
from tqdm import tqdm

CONDITION = "bm25"
RANKINGS_PATH = f"results/rankings/{CONDITION}.parquet"
SPEED_PATH = f"results/speed/{CONDITION}.json"
SEED = 0


def seeded_hash(key):
    return int(hashlib.md5(f"{SEED}:{key}".encode()).hexdigest()[:15], 16)


os.makedirs("results/rankings", exist_ok=True)
os.makedirs("results/speed", exist_ok=True)

/Users/adiram/.local/share/uv/python/cpython-3.11.15-macos-aarch64-none/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
data = pd.read_parquet("data/data.parquet")
splits = pd.read_parquet("data/splits.parquet")
histories = pd.read_parquet("data/histories.parquet")

data["entry"] = data["title"] + " | " + data["url"]
row_of_did = pd.Series(np.arange(len(data)), index=data["answer_did"])
# Tie-break key for every page, so equal BM25 scores always come out in the same order.
did_hash = np.array([seeded_hash(did) for did in tqdm(data["answer_did"], desc="hashing ids")], dtype=np.int64)

eval_rows = (splits[splits["split"].isin(["validation", "test"])]
             .merge(data[["qid", "query", "answer_did", "train_question", "test_question"]], on="qid")
             .merge(histories, on="qid"))
print(eval_rows["split"].value_counts().to_string())
print(f"{len(data):,} entries in the index")
assert eval_rows["test_question"].notna().all(), "validation/test rows are missing Llama questions"
assert eval_rows["history_dids"].str.len().eq(200).all()
assert all(did in row_of_did.index for dids in eval_rows["history_dids"] for did in dids)

hashing ids: 100%|██████████| 300000/300000 [00:00<00:00, 1711169.26it/s]


split
test          2000
validation     500
300,000 entries in the index


In [4]:
# One global index over all entries; the time per entry is the indexing cost reported in the speed file.
stemmer = Stemmer.Stemmer("english")
start = time.perf_counter()
corpus_tokens = bm25s.tokenize(data["entry"].tolist(), stopwords="en", stemmer=stemmer, show_progress=False)
retriever = bm25s.BM25(k1=1.2, b=0.75)
retriever.index(corpus_tokens, show_progress=False)
index_seconds = time.perf_counter() - start
entry_ms = index_seconds * 1000 / len(data)
print(f"indexed {len(data):,} entries in {index_seconds:.1f} s ({entry_ms:.4f} ms per entry), "
      f"vocabulary {len(retriever.vocab_dict):,} terms")

indexed 300,000 entries in 2.5 s (0.0085 ms per entry), vocabulary 254,316 terms


In [5]:
def rank_history(question, history_dids):
    tokens = bm25s.tokenize([question], stopwords="en", stemmer=stemmer, return_ids=False, show_progress=False)[0]
    tokens = [t for t in tokens if t in retriever.vocab_dict]  # words never seen in any entry score nothing
    rows = row_of_did[history_dids].to_numpy()
    if tokens:
        scores = retriever.get_scores(tokens)[rows]
    else:
        scores = np.zeros(len(rows))
    # Sort by score (high first), then by seeded hash of the id.
    order = np.lexsort((did_hash[rows], -scores))
    return [history_dids[i] for i in order]


rank_history(eval_rows["test_question"].iloc[0], list(eval_rows["history_dids"].iloc[0]))[:3]

['D1411859', 'D781651', 'D777536']

In [6]:
# Questions to rank: validation uses Llama questions; test also gets the Qwen rewrite and the original keyword query.
jobs = []
for row in eval_rows.itertuples():
    jobs.append((row.qid, row.split, "llama", row.test_question, row.history_dids))
    if row.split == "test":
        jobs.append((row.qid, row.split, "qwen", row.train_question, row.history_dids))
        jobs.append((row.qid, row.split, "keyword", row.query, row.history_dids))
print(pd.Series([j[1] + " / " + j[2] for j in jobs]).value_counts().to_string())

if os.path.exists(RANKINGS_PATH):
    rankings = pd.read_parquet(RANKINGS_PATH)
    print(f"loaded {len(rankings):,} rankings from {RANKINGS_PATH}")
else:
    records = []
    for qid, split, source, question, history_dids in tqdm(jobs, desc="ranking"):
        ranked = rank_history(question, list(history_dids))
        records.append({"qid": qid, "split": split, "source": source, "ranked_dids": ranked})
    rankings = pd.DataFrame(records)
    rankings.to_parquet(RANKINGS_PATH, index=False)
    print(f"wrote {len(rankings):,} rankings to {RANKINGS_PATH}")
assert len(rankings) == len(jobs)

test / llama          2000
test / qwen           2000
test / keyword        2000
validation / llama     500


ranking: 100%|██████████| 6500/6500 [00:02<00:00, 2926.72it/s]


wrote 6,500 rankings to results/rankings/bm25.parquet


In [7]:
# CPU time per question: tokenize, score the 200 history entries, sort. Uses the global index, as the rankings do.
test_llama = eval_rows[eval_rows["split"] == "test"]
question_times = []
for row in tqdm(test_llama.itertuples(), total=len(test_llama), desc="timing"):
    history_dids = list(row.history_dids)
    start = time.perf_counter()
    rank_history(row.test_question, history_dids)
    question_times.append(time.perf_counter() - start)
question_ms = 1000 * float(np.mean(question_times))

speed = {"entry_ms": entry_ms, "question_ms": question_ms}
with open(SPEED_PATH, "w") as f:
    json.dump(speed, f, indent=2)
print(f"{len(question_times):,} test questions: mean {question_ms:.2f} ms, median {1000 * np.median(question_times):.2f} ms per question")
print(speed)

timing: 100%|██████████| 2000/2000 [00:00<00:00, 2898.66it/s]

2,000 test questions: mean 0.34 ms, median 0.33 ms per question
{'entry_ms': 0.008484105139990181, 'question_ms': 0.3358400090110081}


In [8]:
# Quick sanity check on validation (full metrics come in the comparison step).
validation = (rankings[(rankings["split"] == "validation") & (rankings["source"] == "llama")]
              .merge(eval_rows[["qid", "answer_did", "test_question"]], on="qid"))
validation["answer_rank"] = [list(ranked).index(did) + 1 for ranked, did in zip(validation["ranked_dids"], validation["answer_did"])]
print(f"validation Llama questions: {len(validation)}")
print(f"validation Recall@1: {(validation['answer_rank'] <= 1).mean():.3f}")
print(f"validation Recall@5: {(validation['answer_rank'] <= 5).mean():.3f}")

entry_of_did = pd.Series(data["entry"].to_numpy(), index=data["answer_did"])
for row in validation.head(5).itertuples():
    print(f"\nQ: {row.test_question}")
    print(f"   answer (rank {row.answer_rank}): {entry_of_did[row.answer_did]}")
    for i, did in enumerate(row.ranked_dids[:3], start=1):
        mark = "*" if did == row.answer_did else " "
        print(f"   {mark}{i}. {entry_of_did[did]}")

validation Llama questions: 500
validation Recall@1: 0.476
validation Recall@5: 0.708

Q: How do I use arrays within arrays in JavaScript?
   answer (rank 5): JavaScript Arrays | w3schools.com/js/js_arrays.asp
    1. JavaScript Array Remove an Element | viralpatel.net/blogs/javascript-array-remove-element-js-array-delete-element
    2. How to Add Elements to an Array | dyn-web.com/javascript/arrays/add.php
    3. JavaScript Delete Specific Item From Array | solidlystated.com/scripting/javascript-delete-from-array

Q: What's an example of Pascal's principle that I read about?
   answer (rank 1): Pascal's principle | britannica.com/science/Pascals-principle
   *1. Pascal's principle | britannica.com/science/Pascals-principle
    2. What Are the Principles of Art? - Definition & Examples | study.com/academy/lesson/what-are-the-principles-of-art-definition-examples.html
    3. What is Morality? - Definition, Principles & Examples | study.com/academy/lesson/what-is-morality-definition-princ